In [1]:
import os, time, tracemalloc
import pandas as pd
from pathlib import Path

METHOD = 'privbayes'
EPSILON = 1.0
OUT_DIR = Path('../synthetic_data')
SDG_DIR = Path('.')
OUT_DIR.mkdir(exist_ok=True)

CONTINUOUS_COLS = ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
CATEGORICAL_COLS = ['workclass', 'marital_status', 'occupation', 'relationship',
                    'race', 'sex', 'native_country', 'income']
TARGET_COL = 'income'

In [2]:
df = pd.read_csv('../data/adult_train.csv')
df[CONTINUOUS_COLS] = df[CONTINUOUS_COLS].astype(float)
df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype('category')
N = len(df)
print(f'Training rows: {N}, columns: {df.columns.tolist()}')

Training rows: 21523, columns: ['age', 'workclass', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


In [3]:
try:
    from synthcity.plugins import Plugins
    from synthcity.plugins.core.dataloader import GenericDataLoader

    loader = GenericDataLoader(df, target_column=TARGET_COL)
    plugin = Plugins().get(METHOD, epsilon=EPSILON)

    # --- fit ---
    tracemalloc.start()
    t0 = time.time()
    plugin.fit(loader)

    # --- generate ---
    synthetic = plugin.generate(count=N).dataframe()
    wall_time = round(time.time() - t0, 2)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    peak_mb = round(peak / 1e6, 2)

    synthetic.to_csv(OUT_DIR / f'{METHOD}_synthetic.csv', index=False)
    print(f'Done — {len(synthetic)} rows | {wall_time}s | {peak_mb} MB peak')
    print(synthetic.head())

    # --- overhead log ---
    overhead_path = SDG_DIR / 'computational_overhead.csv'
    row = pd.DataFrame([{'method': METHOD, 'rows_generated': len(synthetic),
                          'wall_time_s': wall_time, 'peak_memory_mb': peak_mb}])
    if overhead_path.exists():
        existing = pd.read_csv(overhead_path)
        existing = existing[existing['method'] != METHOD]
        row = pd.concat([existing, row], ignore_index=True)
    row.to_csv(overhead_path, index=False)

except Exception as e:
    import traceback
    log = f'ERROR running {METHOD} (epsilon={EPSILON}):\n{traceback.format_exc()}'
    print(log)
    (SDG_DIR / f'{METHOD}_logs.txt').write_text(log)

/opt/anaconda3/envs/priv-sdg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


[2026-06-27T17:31:32.986305+1000][70907][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-06-27T17:31:32.988384+1000][70907][CRITICAL] module disabled: /opt/anaconda3/envs/priv-sdg/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
Generating for node: capital_gain:   0%|          | 0/13 [00:00<?, ?it/s]06/27/2026 17:34:17:WARNING:Probability values don't exactly sum to 1. Differ by: -2.220446049250313e-16. Adjusting values.
06/27/2026 17:34:17:WARNING:Probability values don't exactly sum to 1. Differ by: -2.220446049250313e-16. Adjusting values.
Generating for node: occupation:   0%|          | 0/13 [00:00<?, ?it/s]  06/27/2026 17:34:17:WARNING:Probability values don't exactly sum to 1. Differ by: 1.1102230246251565e-16. Adjusting values.
06/27/2026 17:34:17:WARNING:Probability values don't exactly sum to 1. Differ by: -2.220446049250313e-16. Adjusting values.
06/27/2026 17:34:17:WARNING:Probability values don't exactly sum to 1. Differ by: 1.1

Done — 21523 rows | 165.71s | 21.52 MB peak
    age         workclass  education_num      marital_status  \
0  35.0           Private            9.0  Married-civ-spouse   
1  61.0         Local-gov            9.0            Divorced   
2  50.0           Private           12.0            Divorced   
3  48.0           Private            2.0  Married-civ-spouse   
4  43.0  Self-emp-not-inc           13.0  Married-civ-spouse   

         occupation   relationship   race     sex  capital_gain  capital_loss  \
0      Adm-clerical           Wife  White  Female           0.0           0.0   
1  Transport-moving  Not-in-family  White    Male           0.0           0.0   
2             Sales      Own-child  White    Male           0.0        1668.0   
3      Adm-clerical        Husband  White    Male           0.0           0.0   
4      Craft-repair        Husband  White    Male           0.0           0.0   

   hours_per_week native_country income  
0            40.0  United-States   >50K  
